# Exploración de Cloud Provider Analytics

Ramón Ojea · Primera entrega. La evidencia usa Python estándar. No se implementa aquí el pipeline Spark/Cassandra.

In [1]:
from pathlib import Path
import sys, json
root = Path.cwd()
if not (root / 'src/exploracion.py').exists():
    root = root.parent
assert (root / 'src/exploracion.py').exists(), 'Ejecutar desde la raíz del repositorio o notebooks/'
sys.path.insert(0, str(root / 'src'))
from exploracion import load, quality, watermark_simulation, mapreduce
tables, batches = load(root / 'data/datalake/landing')
print({name: len(rows) for name, rows in tables.items()})
print('Archivos:', len(batches), 'Eventos:', sum(len(rows) for _, rows in batches))

{'customers_orgs': 80, 'users': 800, 'resources': 400, 'support_tickets': 1000, 'marketing_touches': 1500, 'nps_surveys': 92, 'billing_monthly': 240}
Archivos: 120 Eventos: 43200


## Tipos originales y evolución de esquema

Leemos JSON original para distinguir números de strings; una lectura que convierte todos los primitivos a texto oculta esa diferencia.

In [2]:
from collections import Counter
events = [e for _, rows in batches for e in rows]
print('Tipos de value:', dict(Counter(type(e['value']).__name__ for e in events)))
for version in sorted({e['schema_version'] for e in events}):
    rows = [e for e in events if e['schema_version'] == version]
    print(version, len(rows), min(e['timestamp'] for e in rows), max(e['timestamp'] for e in rows))

Tipos de value: {'NoneType': 877, 'float': 41014, 'str': 1309}
1 10800 2025-07-03T00:02:00Z 2025-07-17T23:56:00Z
2 32400 2025-07-18T00:01:00Z 2025-08-31T23:58:00Z


## Calidad

Flags y cuarentena son propuestas. Los nulos y negativos no se eliminan de los originales. La escala CSAT debe confirmarse.

In [3]:
for issue in quality(tables, batches):
    if issue['count'] or issue['rule'] == 'invalid_nonempty_tags_json':
        print(issue['source'], issue['rule'], issue['count'], '/', issue['total'])

usage_events value_string 1309 / 43200
usage_events value_null 877 / 43200
usage_events unit_null 2075 / 43200
usage_events negative_cost 216 / 43200
usage_events cost_gt_100_usd 48 / 43200
resources invalid_nonempty_tags_json 0 / 317
resources tags_null 83 / 400
billing_monthly negative_subtotal 13 / 240
billing_monthly credits_null 137 / 240
billing_monthly usd_fx_not_one 160 / 240
users login_before_created 232 / 800
customers_orgs nps_null 11 / 80
customers_orgs nps_outside_minus100_100 1 / 80
nps_surveys nps_null 19 / 92
support_tickets open_ticket 240 / 1000
support_tickets csat_null 254 / 1000
support_tickets csat_outside_1_5 40 / 1000


## Desorden temporal y watermark

Un archivo por lote y orden lexicográfico. Esto estima eventos debajo de un umbral; no ejecuta un stream Spark ni mide el descarte real.

In [4]:
counts = [len({e['timestamp'][:10] for e in rows}) for _, rows in batches]
print('Fechas por archivo:', min(counts), 'a', max(counts))
for row in watermark_simulation(batches):
    print(row)

Fechas por archivo: 59 a 60
{'margin_hours': 1, 'below_threshold': 42801, 'total': 43200, 'percent': 99.076}
{'margin_hours': 2, 'below_threshold': 42783, 'total': 43200, 'percent': 99.035}
{'margin_hours': 24, 'below_threshold': 42118, 'total': 43200, 'percent': 97.495}
{'margin_hours': 168, 'below_threshold': 37831, 'total': 43200, 'percent': 87.572}
{'margin_hours': 720, 'below_threshold': 21421, 'total': 43200, 'percent': 49.586}
{'margin_hours': 1440, 'below_threshold': 0, 'total': 43200, 'percent': 0.0}


## MapReduce y reconciliación

Map, suma local, agrupación conceptual y reduce. Se verifica el costo de cada clave con agregación directa; esta salida es referencia para la implementación posterior.

In [5]:
mart, verification = mapreduce(batches)
print(json.dumps(verification, indent=2))
print('Primeras dos filas:')
for row in mart[:2]:
    print(row)

{
  "mapped_pairs": 43200,
  "combined_pairs": 42572,
  "gold_keys": 11050,
  "total_cost_usd": "147433.9778",
  "validation": "cost_by_key_and_event_count_ok"
}
Primeras dos filas:
{'org_id': 'org_0lvsnujz', 'usage_date': '2025-07-03', 'service': 'analytics', 'daily_cost_usd': '8.8739', 'requests_known': '116.0', 'cpu_hours_known': '4.6168', 'storage_gb_hours_known': '12.7808', 'event_count': '3', 'missing_value_count': '0', 'negative_cost_count': '0'}
{'org_id': 'org_0lvsnujz', 'usage_date': '2025-07-03', 'service': 'database', 'daily_cost_usd': '0.5723', 'requests_known': '0', 'cpu_hours_known': '1.7515', 'storage_gb_hours_known': '14.7891', 'event_count': '3', 'missing_value_count': '0', 'negative_cost_count': '0'}


## Conclusión

La llegada del histórico requiere captura sin pérdida y reconciliación. Los tipos ambiguos requieren conservar raw y castear con control. El dataset no exige un cluster por tamaño. PySpark, streaming y serving se implementarán en la segunda entrega.